In [3]:
import pandas as pd

historical = pd.read_csv("MPS_Borough_Level_Crime_Historical.csv")
recent = pd.read_csv("MPS_Borough_Level_Crime_Recent_24_month.csv")
newest = pd.read_csv("MPS_Borough_Level_Crime _Most_Recent_24_months_new.csv")

id_cols_old = ["MajorText", "MinorText", "BoroughName"]
hist_long = historical.melt(id_vars=id_cols_old, var_name="YearMonth", value_name="CrimeCount")
recent_long = recent.melt(id_vars=id_cols_old, var_name="YearMonth", value_name="CrimeCount")

id_cols_new = ["Group", "SubGroup", "BOCU"]
newest_long = newest.melt(id_vars=id_cols_new, var_name="YearMonth", value_name="CrimeCount")
newest_long = newest_long.rename(columns={"BOCU": "BoroughName", "Group": "MajorText", "SubGroup": "MinorText"})

crime = pd.concat([hist_long, recent_long, newest_long], ignore_index=True)

not_boroughs = ["Unknown", "London Heathrow and London City Airports"]
crime = crime[~crime["BoroughName"].isin(not_boroughs)]
crime["Date"] = pd.to_datetime(crime["YearMonth"], format="%Y%m")

# dedupe on the FULL combination (borough + category + date), not just borough + date
crime = crime.sort_values("Date").drop_duplicates(subset=["BoroughName", "MajorText", "MinorText", "Date"], keep="last")

camden = crime[crime["BoroughName"] == "Camden"].groupby("Date")["CrimeCount"].sum().sort_index()
camden = camden.asfreq("MS")

print(camden.shape)
print(camden.tail(8))

(195,)
Date
2025-11-01    3597
2025-12-01    3381
2026-01-01    2986
2026-02-01    3120
2026-03-01    3282
2026-04-01    3131
2026-05-01    3366
2026-06-01    3380
Freq: MS, Name: CrimeCount, dtype: int64


In [4]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import numpy as np

train = camden[:"2026-02-01"]
test = camden["2026-03-01":"2026-06-01"]

print("Train size:", len(train))
print("Test size:", len(test))

model = SARIMAX(train, order=(1, 1, 0), seasonal_order=(1, 1, 0, 12))
fitted = model.fit(disp=False)

forecast_result = fitted.get_forecast(steps=4)
forecast = forecast_result.predicted_mean
conf_int = forecast_result.conf_int(alpha=0.05)

comparison = pd.DataFrame({
    "Actual": test.values,
    "Forecast": forecast.values.round(0),
    "Lower_95CI": conf_int.iloc[:, 0].values.round(0),
    "Upper_95CI": conf_int.iloc[:, 1].values.round(0)
}, index=test.index)

comparison["Error"] = comparison["Actual"] - comparison["Forecast"]
comparison["Pct_Error"] = (comparison["Error"].abs() / comparison["Actual"] * 100).round(1)

print(comparison)
print()
print("Mean % error:", comparison["Pct_Error"].mean().round(2))

Train size: 191
Test size: 4
            Actual  Forecast  Lower_95CI  Upper_95CI  Error  Pct_Error
Date                                                                  
2026-03-01    3282    3144.0      2655.0      3633.0  138.0        4.2
2026-04-01    3131    3000.0      2341.0      3659.0  131.0        4.2
2026-05-01    3366    3060.0      2264.0      3856.0  306.0        9.1
2026-06-01    3380    3142.0      2230.0      4054.0  238.0        7.0

Mean % error: 6.12
